# Find a Tender 

This notebooks scrapes find a tender for relevent results and uploads it to notion. 


In [1]:
import requests
import pandas as pd
from decimal import Decimal
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional, Set

BASE_URL = "https://www.find-tender.service.gov.uk/api/1.0/ocdsReleasePackages"

# ---- time window (3 days, UTC) ----
_now_utc = datetime.now(timezone.utc).replace(microsecond=0)
_UPDATED_TO = _now_utc.strftime("%Y-%m-%dT%H:%M:%S")
_UPDATED_FROM = (_now_utc - timedelta(days=2)).strftime("%Y-%m-%dT%H:%M:%S")
# -----------------------------------
# -----------------------------------


# Target CPV codes (8-digit, as strings)
TARGET_CPV: Set[str] = {
    "66171000",  # Financial consultancy services
    "73000000",  # Research and development services and related consultancy services
    "73100000",  # Research and experimental development services
    "73110000",  # Research services
    "73120000",  # Experimental development services
    "73200000",  # Research and development consultancy services
    "73210000",  # Research consultancy services
    "73220000",  # Development consultancy services
    "73300000",  # Design and execution of research and development
    "73400000",  # Research and Development services on security and defence materials
    "75210000",  # Foreign affairs and other services
    "75211200",  # Foreign economic-aid-related services
    "79311100",  # Survey design services
    "79311300",  # Survey analysis services
    "79311400",  # Economic research services
    "79311410",  # Economic impact assessment
    "79313000",  # Performance review services
    "79314000",  # Feasibility study
    "79315000",  # Social research services
    "79320000",  # Public-opinion polling services
    "79330000",  # Statistical services
    "79411000",  # General management consultancy services
    "79411100",  # Business development consultancy service
    "79419000",  # Evaluation consultancy services
    "90713000",  # Environmental issues consultancy services
    "98200000",  # Equal opportunities consultancy services
    "80000000"   # Education and training services
}
# ---- robust pagination with NO server-side stage filtering ----
from datetime import datetime, timedelta
from typing import Optional, Tuple, List, Dict, Any
import requests

import time

MAX_PER_PAGE = 50
MAX_FETCH_RETRIES = 8
SUCCESS_PAUSE_SECONDS = 0.35
MIN_SPLIT_SECONDS = 15 * 60  # do not keep splitting below 15 minutes

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "LSE-Consulting-Find-a-Tender/1.0"})

RUN_STARTED_AT = datetime.now(timezone.utc)

FETCH_STATS = {
    "requests": 0,
    "pages_ok": 0,
    "retries_429": 0,
    "retries_503": 0,
    "top_level_slices": 0,
    "split_slices": 0,
    "raw_releases_seen": 0,
}

def log(msg: str) -> None:
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    print(f"[Find a Tender] {ts} | {msg}", flush=True)

log(f"Run started | window={_UPDATED_FROM} -> {_UPDATED_TO} | per_page={MAX_PER_PAGE}")

def _iso_no_z(dt: datetime) -> str:
    # API expects 'YYYY-MM-DDTHH:MM:SS' (UTC, no 'Z')
    return dt.strftime("%Y-%m-%dT%H:%M:%S")

def _sleep_seconds_from_response(r: requests.Response, attempt: int) -> int:
    retry_after = r.headers.get("Retry-After")
    try:
        retry_after_int = int(retry_after) if retry_after else 0
    except ValueError:
        retry_after_int = 0
    return max(retry_after_int, min(60, 2 ** attempt))

def _fetch_page(updated_from: str,
                updated_to: str,
                cursor: Optional[str] = None,
                per_page: int = MAX_PER_PAGE) -> Tuple[List[Dict[str, Any]], Optional[str]]:
    """
    Fetch a single page, retrying politely on 429 / 503.
    No 'stages' filter is sent, to avoid excluding tenderUpdate, etc.
    """
    params = {
        "limit": min(per_page, MAX_PER_PAGE),
        "updatedFrom": updated_from,
        "updatedTo": updated_to,
    }
    if cursor:
        params["cursor"] = cursor

    last_error = None

    for attempt in range(MAX_FETCH_RETRIES):
        FETCH_STATS["requests"] += 1
        log(
            f"Requesting page | attempt={attempt + 1}/{MAX_FETCH_RETRIES} "
            f"| from={updated_from} | to={updated_to} | cursor={'yes' if cursor else 'no'}"
        )

        try:
            r = SESSION.get(BASE_URL, params=params, timeout=60)
        except requests.RequestException as e:
            last_error = e
            sleep_for = min(60, 2 ** attempt)
            log(f"Request error | sleeping {sleep_for}s | error={e}")
            time.sleep(sleep_for)
            continue

        if r.status_code == 400:
            log(f"Received 400 for window {updated_from} -> {updated_to}; returning empty page")
            return [], None

        if r.status_code == 429:
            FETCH_STATS["retries_429"] += 1
            sleep_for = _sleep_seconds_from_response(r, attempt)
            log(f"Received 429 rate limit | sleeping {sleep_for}s then retrying")
            time.sleep(sleep_for)
            continue

        if r.status_code == 503:
            FETCH_STATS["retries_503"] += 1
            sleep_for = _sleep_seconds_from_response(r, attempt)
            log(f"Received 503 service unavailable | sleeping {sleep_for}s then retrying")
            time.sleep(sleep_for)
            continue

        r.raise_for_status()
        payload = r.json() or {}
        releases = payload.get("releases") or []
        next_cursor = payload.get("cursor")

        FETCH_STATS["pages_ok"] += 1
        FETCH_STATS["raw_releases_seen"] += len(releases)

        log(
            f"Page OK | releases={len(releases)} | next_cursor={'yes' if next_cursor else 'no'} "
            f"| total_pages={FETCH_STATS['pages_ok']} | total_raw_releases={FETCH_STATS['raw_releases_seen']}"
        )

        time.sleep(SUCCESS_PAUSE_SECONDS)
        return releases, next_cursor

    if last_error is not None:
        raise last_error
    raise requests.HTTPError(f"Failed after {MAX_FETCH_RETRIES} retries for params={params}")
    


def _walk_slice(u_from: str,
                u_to: str,
                per_page: int = MAX_PER_PAGE,
                min_split_seconds: int = MIN_SPLIT_SECONDS):
    """
    Yield all releases in [u_from, u_to], ensuring u_to > u_from.
    If a slice saturates (len == per_page and no cursor), split the time window,
    but stop splitting once the window gets very small to avoid request explosions.
    """
    dt_from = datetime.fromisoformat(u_from)
    dt_to = datetime.fromisoformat(u_to)

    if dt_to <= dt_from:
        log(f"Skipping empty slice | {u_from} -> {u_to}")
        return

    log(f"Walking slice | {u_from} -> {u_to}")

    releases, cursor = _fetch_page(u_from, u_to, cursor=None, per_page=per_page)
    for rel in releases:
        if isinstance(rel, dict):
            yield rel

    page_num = 1
    while cursor:
        page_num += 1
        log(f"Following cursor | slice={u_from} -> {u_to} | page={page_num}")
        page, cursor = _fetch_page(u_from, u_to, cursor=cursor, per_page=per_page)
        if not page:
            log(f"Cursor returned empty page | slice={u_from} -> {u_to}")
            break
        for rel in page:
            if isinstance(rel, dict):
                yield rel

    window_seconds = (dt_to - dt_from).total_seconds()

    if (not cursor) and len(releases) >= per_page and window_seconds > min_split_seconds:
        FETCH_STATS["split_slices"] += 1
        mid = dt_from + (dt_to - dt_from) / 2
        log(
            f"Slice saturated with {len(releases)} first-page releases and no cursor; "
            f"splitting | {u_from} -> {u_to}"
        )
        yield from _walk_slice(_iso_no_z(dt_from), _iso_no_z(mid), per_page=per_page, min_split_seconds=min_split_seconds)
        yield from _walk_slice(_iso_no_z(mid), _iso_no_z(dt_to), per_page=per_page, min_split_seconds=min_split_seconds)


def iter_releases_window(updated_from: str,
                         updated_to: str,
                         per_page: int = MAX_PER_PAGE,
                         slice_hours: int = 2):
    dt_from = datetime.fromisoformat(updated_from)
    dt_to = datetime.fromisoformat(updated_to)
    if dt_to <= dt_from:
        log(f"Invalid window | updated_from={updated_from} | updated_to={updated_to}")
        return

    step = timedelta(hours=slice_hours)
    slice_start = dt_from
    slice_idx = 0

    while slice_start < dt_to:
        slice_end = min(slice_start + step, dt_to)
        slice_idx += 1
        FETCH_STATS["top_level_slices"] += 1
        log(f"Starting top-level slice {slice_idx} | {_iso_no_z(slice_start)} -> {_iso_no_z(slice_end)}")
        yield from _walk_slice(_iso_no_z(slice_start), _iso_no_z(slice_end), per_page=per_page)
        log(f"Finished top-level slice {slice_idx} | {_iso_no_z(slice_start)} -> {_iso_no_z(slice_end)}")
        slice_start = slice_end

# ---------------------------------------------------------------------

def _parse_iso(dt: Optional[str]) -> datetime:
    return datetime.fromisoformat(dt.replace("Z", "+00:00")) if dt else datetime.min

def _fmt_date(d: Optional[str]) -> Optional[str]:
    if not d:
        return None
    try:
        dt = datetime.fromisoformat(d.replace("Z", "+00:00"))
        return dt.strftime("%d %B %Y").lstrip("0")
    except Exception:
        return d

def extract_buyer_party(release: Dict[str, Any]) -> Dict[str, Any]:
    buyer_ref = release.get("buyer", {})
    for p in release.get("parties", []) or []:
        if buyer_ref and p.get("id") == buyer_ref.get("id") and "buyer" in (p.get("roles") or []):
            return p
    for p in release.get("parties", []) or []:
        if "buyer" in (p.get("roles") or []):
            return p
    return {}

def extract_employer_name(release: Dict[str, Any]) -> Optional[str]:
    return release.get("buyer", {}).get("name") or extract_buyer_party(release).get("name")

def extract_employer_url(release: Dict[str, Any]) -> Optional[str]:
    return (extract_buyer_party(release).get("details") or {}).get("url")

def extract_title(release: Dict[str, Any]) -> Optional[str]:
    return (release.get("tender") or {}).get("title")

def extract_language(release: Dict[str, Any]) -> Optional[str]:
    return release.get("language")

def extract_tag_list(release: Dict[str, Any]) -> List[str]:
    return release.get("tag") or []

def extract_tag(release: Dict[str, Any]) -> str:
    return ", ".join(extract_tag_list(release))

def extract_description(release: Dict[str, Any]) -> Optional[str]:
    tender = release.get("tender") or {}
    if tender.get("description"):
        return tender["description"]
    planning_docs = (release.get("planning") or {}).get("documents") or []
    if planning_docs:
        return planning_docs[0].get("description")
    for a in release.get("awards") or []:
        if a.get("description"):
            return a["description"]
    return None

def _best_amount(v: Optional[Dict[str, Any]]) -> Optional[Decimal]:
    if not v:
        return None
    if v.get("amountGross") is not None:
        return Decimal(str(v["amountGross"]))
    if v.get("amount") is not None:
        return Decimal(str(v["amount"]))
    return None

def extract_value(release: Dict[str, Any]) -> Dict[str, Any]:
    # 1) contracts
    for c in release.get("contracts") or []:
        amt = _best_amount(c.get("value"))
        if amt is not None:
            return {"amount": amt, "currency": (c.get("value") or {}).get("currency"), "source": "contracts"}
    # 2) tender.lots
    tender = release.get("tender") or {}
    for lot in tender.get("lots") or []:
        amt = _best_amount(lot.get("value"))
        if amt is not None:
            return {"amount": amt, "currency": (lot.get("value") or {}).get("currency"), "source": "tender.lot"}
    # 3) tender.value
    amt = _best_amount(tender.get("value"))
    if amt is not None:
        return {"amount": amt, "currency": (tender.get("value") or {}).get("currency"), "source": "tender"}
    # 4) awards
    for a in release.get("awards") or []:
        amt = _best_amount(a.get("value"))
        if amt is not None:
            return {"amount": amt, "currency": (a.get("value") or {}).get("currency"), "source": "awards"}
    return {"amount": None, "currency": None, "source": "none"}

def extract_deadline_enquiry(release: Dict[str, Any]) -> Optional[str]:
    """
    Application deadline is the ENQUIRY deadline, not tender deadline.
    If missing, return None.
    """
    tender = release.get("tender") or {}
    ep = tender.get("enquiryPeriod") or {}
    end = ep.get("endDate")
    return end or None

def _iter_all_documents(release: Dict[str, Any]) -> List[Dict[str, Any]]:
    docs: List[Dict[str, Any]] = []
    tender = release.get("tender") or {}
    docs.extend(tender.get("documents") or [])
    for a in release.get("awards") or []:
        docs.extend(a.get("documents") or [])
    for c in release.get("contracts") or []:
        docs.extend(c.get("documents") or [])
    planning = release.get("planning") or {}
    docs.extend(planning.get("documents") or [])
    return docs

def extract_notice_url(release: Dict[str, Any]) -> Optional[str]:
    docs = [d for d in _iter_all_documents(release) if isinstance(d, dict) and d.get("url")]
    if not docs:
        return None
    with_dates = [(d, _parse_iso(d.get("datePublished"))) for d in docs if d.get("datePublished")]
    if with_dates:
        with_dates.sort(key=lambda x: x[1], reverse=True)
        return with_dates[0][0]["url"]
    return docs[0]["url"]

def gather_cpv_codes(release: Dict[str, Any]) -> Set[str]:
    cpvs: Set[str] = set()
    tender = release.get("tender") or {}

    # tender.classification
    t_class = tender.get("classification") or {}
    if t_class.get("scheme") == "CPV" and t_class.get("id"):
        cpvs.add(str(t_class["id"]))

    # tender.items classifications
    for item in tender.get("items") or []:
        cls = item.get("classification") or {}
        if cls.get("scheme") == "CPV" and cls.get("id"):
            cpvs.add(str(cls["id"]))
        for add in item.get("additionalClassifications") or []:
            if add.get("scheme") == "CPV" and add.get("id"):
                cpvs.add(str(add["id"]))

    # awards/contract items just in case CPVs are only there
    for a in release.get("awards") or []:
        for item in a.get("items") or []:
            cls = item.get("classification") or {}
            if cls.get("scheme") == "CPV" and cls.get("id"):
                cpvs.add(str(cls["id"]))
            for add in item.get("additionalClassifications") or []:
                if add.get("scheme") == "CPV" and add.get("id"):
                    cpvs.add(str(add["id"]))

    for c in release.get("contracts") or []:
        for item in c.get("items") or []:
            cls = item.get("classification") or {}
            if cls.get("scheme") == "CPV" and cls.get("id"):
                cpvs.add(str(cls["id"]))
            for add in item.get("additionalClassifications") or []:
                if add.get("scheme") == "CPV" and add.get("id"):
                    cpvs.add(str(add["id"]))
    return cpvs

def extract_countries(release: Dict[str, Any]) -> List[str]:
    """
    Return human-readable country names from delivery addresses
    across tender, awards, contracts. Deduplicated, order-stable.
    """
    countries: List[str] = []

    def add_country(val: Optional[str]):
        if val and val not in countries:
            countries.append(val)

    tender = release.get("tender") or {}
    for item in tender.get("items") or []:
        for da in item.get("deliveryAddresses") or []:
            add_country(da.get("countryName") or da.get("country"))

    for a in release.get("awards") or []:
        for item in a.get("items") or []:
            for da in item.get("deliveryAddresses") or []:
                add_country(da.get("countryName") or da.get("country"))

    for c in release.get("contracts") or []:
        for item in c.get("items") or []:
            for da in item.get("deliveryAddresses") or []:
                add_country(da.get("countryName") or da.get("country"))

    return countries

def extract_contract_dates_text(release: Dict[str, Any]) -> Optional[str]:
    parts: List[str] = []
    for c in release.get("contracts") or []:
        period = c.get("period") or {}
        s, e = _fmt_date(period.get("startDate")), _fmt_date(period.get("endDate"))
        if s and e:
            parts.append(f"{s} to {e}")
        elif s:
            parts.append(f"From {s}")
        elif e:
            parts.append(f"Until {e}")

    tender = release.get("tender") or {}
    for lot in tender.get("lots") or []:
        cp = lot.get("contractPeriod") or {}
        s2, e2 = _fmt_date(cp.get("startDate")), _fmt_date(cp.get("endDate"))
        if s2 and e2:
            parts.append(f"{s2} to {e2}")
        elif s2:
            parts.append(f"From {s2}")
        elif e2:
            parts.append(f"Until {e2}")
        max_ext = _fmt_date(cp.get("maxExtentDate"))
        if max_ext:
            parts.append(f"Possible extension to {max_ext}")
        if lot.get("hasRenewal"):
            renewal = lot.get("renewal") or {}
            if renewal.get("description"):
                parts.append(f"Description of possible extension: {renewal['description']}")

    if not parts:
        return None
    seen = set()
    uniq = []
    for p in parts:
        if p and p not in seen:
            uniq.append(p); seen.add(p)
    return " | ".join(uniq)

# --------------------- fetch, FILTER, and build rows ---------------------
log("Starting raw release collection")
rels: List[Dict[str, Any]] = []
for i, rel in enumerate(iter_releases_window(_UPDATED_FROM, _UPDATED_TO, per_page=50, slice_hours=2), start=1):
    rels.append(rel)
    if i % 100 == 0:
        log(f"Collected {i} raw releases so far")

log(f"Finished raw release collection | total_raw_releases={len(rels)}")

log("Starting filtering and row building")
rows = []
skipped_award_or_contract = 0
skipped_no_target_cpv = 0

for i, rel in enumerate(rels, start=1):
    tags = set(extract_tag_list(rel))
    # Exclude award/contract-tagged releases
    if "award" in tags or "contract" in tags:
        skipped_award_or_contract += 1
        continue
    # CPV filter: only keep releases that mention at least one target CPV
    cpvs = gather_cpv_codes(rel)
    if not (cpvs & TARGET_CPV):
        continue

    if not (cpvs & TARGET_CPV):
        skipped_no_target_cpv += 1
        continue

    val = extract_value(rel)
    countries = extract_countries(rel)

    rows.append({
        "ocid": rel.get("ocid"),
        "release_id": rel.get("id"),
        "release_date": rel.get("date"),
        "tag": ", ".join(sorted(tags)),
        "language": extract_language(rel),

        "employer_name": extract_employer_name(rel),
        "employer_website": extract_employer_url(rel),

        "title": extract_title(rel),
        "description_most_recent": extract_description(rel),

        # Find a Tender notice URL and ENQUIRY deadline
        "find_a_tender_url": extract_notice_url(rel),
        "application_deadline_iso": extract_deadline_enquiry(rel),

        # Countries of contract delivery (semicolon-joined for display)
        "locations": "; ".join(countries) if countries else None,

        # Contract dates (plain text)
        "contract_dates_text": extract_contract_dates_text(rel),

        # CPVs matched (for debugging/visibility)
        "cpv_codes": "; ".join(sorted(cpvs)) if cpvs else None,

        # Value: keep exact Decimal and a pretty string
        "value_amount_decimal": val["amount"],
        "value_amount": f"{val['amount']:,.0f}" if val["amount"] is not None else None,
        "value_currency": val["currency"],
        "value_source": val["source"],
    })

    if i % 100 == 0:
        log(
            f"Processed {i}/{len(rels)} raw releases | kept={len(rows)} "
            f"| skipped_award_contract={skipped_award_or_contract} "
            f"| skipped_no_target_cpv={skipped_no_target_cpv}"
        )


df = pd.DataFrame(rows).sort_values(["release_date", "ocid"], ascending=[False, True]).reset_index(drop=True)

log(
    f"Run complete | final_rows={len(df)} | requests={FETCH_STATS['requests']} "
    f"| pages_ok={FETCH_STATS['pages_ok']} | retries_429={FETCH_STATS['retries_429']} "
    f"| retries_503={FETCH_STATS['retries_503']} | top_level_slices={FETCH_STATS['top_level_slices']} "
    f"| split_slices={FETCH_STATS['split_slices']} | raw_releases_seen={FETCH_STATS['raw_releases_seen']}"
)

# Display-friendly floats
pd.set_option("display.float_format", lambda x: f"{x:,.0f}")
df


[Find a Tender] 2026-07-24 11:42:36 UTC | Run started | window=2026-07-22T11:42:36 -> 2026-07-24T11:42:36 | per_page=50


[Find a Tender] 2026-07-24 11:42:36 UTC | Starting raw release collection


[Find a Tender] 2026-07-24 11:42:36 UTC | Starting top-level slice 1 | 2026-07-22T11:42:36 -> 2026-07-22T13:42:36


[Find a Tender] 2026-07-24 11:42:36 UTC | Walking slice | 2026-07-22T11:42:36 -> 2026-07-22T13:42:36


[Find a Tender] 2026-07-24 11:42:36 UTC | Requesting page | attempt=1/8 | from=2026-07-22T11:42:36 | to=2026-07-22T13:42:36 | cursor=no


...

[Find a Tender] 2026-07-24 11:53:20 UTC | Finished top-level slice 24 | 2026-07-24T09:42:36 -> 2026-07-24T11:42:36


[Find a Tender] 2026-07-24 11:53:20 UTC | Finished raw release collection | total_raw_releases=2357


[Find a Tender] 2026-07-24 11:53:20 UTC | Starting filtering and row building


[Find a Tender] 2026-07-24 11:53:20 UTC | Run complete | final_rows=35 | requests=79 | pages_ok=74 | retries_429=5 | retries_503=0 | top_level_slices=24 | split_slices=25 | raw_releases_seen=2357


,ocid,release_id,release_date,tag,language,employer_name,employer_website,title,description_most_recent,find_a_tender_url,application_deadline_iso,locations,contract_dates_text,cpv_codes,value_amount_decimal,value_amount,value_currency,value_source
0,ocds-h6vhtk-06d059,070155-2026,2026-07-24T11:02:55+01:00,tenderUpdate,en,Natural England,None,Exposure and Effects of Veterinary Medicines a...,Passive quantitative sampling methodology will...,https://www.find-tender.service.gov.uk/Notice/...,2026-08-20T12:00:00+01:00,United Kingdom,4 September 2026 to 31 March 2027,73000000,50000,"50,000",GBP,tender.lot
1,ocds-h6vhtk-06d059,070155-2026,2026-07-24T11:02:55+01:00,tenderUpdate,en,Natural England,None,Exposure and Effects of Veterinary Medicines a...,Passive quantitative sampling methodology will...,https://www.find-tender.service.gov.uk/Notice/...,2026-08-20T12:00:00+01:00,United Kingdom,4 September 2026 to 31 March 2027,73000000,50000,"50,000",GBP,tender.lot
2,ocds-h6vhtk-06d059,070155-2026,2026-07-24T11:02:55+01:00,tenderUpdate,en,Natural England,None,Exposure and Effects of Veterinary Medicines a...,Passive quantitative sampling methodology will...,https://www.find-tender.service.gov.uk/Notice/...,2026-08-20T12:00:00+01:00,United Kingdom,4 September 2026 to 31 March 2027,73000000,50000,"50,000",GBP,tender.lot
3,ocds-h6vhtk-06d2f4,070151-2026,2026-07-24T11:00:04+01:00,tender,en,"Newry, Mourne and Down District Council (NMD)",None,RFQ 2026/25 - NMANDD PEACEPLUS Thriving Togeth...,Competent and creative suppliers are invited t...,https://www.find-tender.service.gov.uk/Notice/...,2026-08-17T12:00:00+01:00,United Kingdom,11 September 2026 to 31 December 2026,80000000; 92000000; 98000000,23454.0,"23,454",GBP,tender.lot
4,ocds-h6vhtk-06d2f4,070151-2026,2026-07-24T11:00:04+01:00,tender,en,"Newry, Mourne and Down District Council (NMD)",None,RFQ 2026/25 - NMANDD PEACEPLUS Thriving Togeth...,Competent and creative suppliers are invited t...,https://www.find-tender.service.gov.uk/Notice/...,2026-08-17T12:00:00+01:00,United Kingdom,11 September 2026 to 31 December 2026,80000000; 92000000; 98000000,23454.0,"23,454",GBP,tender.lot
5,ocds-h6vhtk-06d2f4,070151-2026,2026-07-24T11:00:04+01:00,tender,en,"Newry, Mourne and Down District Council (NMD)",None,RFQ 2026/25 - NMANDD PEACEPLUS Thriving Togeth...,Competent and creative suppliers are invited t...,https://www.find-tender.service.gov.uk/Notice/...,2026-08-17T12:00:00+01:00,United Kingdom,11 September 2026 to 31 December 2026,80000000; 92000000; 98000000,23454.0,"23,454",GBP,tender.lot
6,ocds-h6vhtk-06d290,069976-2026,2026-07-23T20:17:06+01:00,planning,en,Food Standards Agency,https://www.food.gov.uk/,Effect of probiotics on Campylobacter colonisa...,A key component of the FSA work is to reduce t...,https://www.find-tender.service.gov.uk/Notice/...,2026-09-30T12:00:00+01:00,None,1 January 2027 to 31 December 2028 | Possible ...,73000000; 73100000; 73110000; 73111000; 85200000,600000.0,"600,000",GBP,tender.lot
7,ocds-h6vhtk-06d1c1,069649-2026,2026-07-23T12:23:11+01:00,tender,en,Liverpool School of Tropical Medicine,None,ET190 Consultancy - Capacity Strenthening,The purpose of this project is to seek a consu...,https://www.find-tender.service.gov.uk/Notice/...,2026-08-07T17:00:00+01:00,United Kingdom; France,7 September 2026 to 31 May 2028,73210000; 79411000; 79419000,117600.0,"117,600",GBP,tender.lot
8,ocds-h6vhtk-06d1c1,069649-2026,2026-07-23T12:23:11+01:00,tender,en,Liverpool School of Tropical Medicine,None,ET190 Consultancy - Capacity Strenthening,The purpose of this project is to seek a consu...,https://www.find-tender.service.gov.uk/Notice/...,2026-08-07T17:00:00+01:00,United Kingdom; France,7 September 2026 to 31 May 2028,73210000; 79411000; 79419000,117600.0,"117,600",GBP,tender.lot
9,ocds-h6vhtk-06cf88,069503-2026,2026-07-23T10:13:09+01:00,tenderUpdate,en,Oxford Health NHS Foundation Trust,https://www.oxfordhealth.nhs.uk/,Neurodiverse Environmental Audits,Environment Audit of Low and M

# Upload to notion

In [2]:
# ---------- Notion upload for Find a Tender (exact schema match) ----------
# Prereqs: df already exists with the columns built above

import os
from datetime import datetime, timezone

# Notion credentials (use the ones you provided)
NOTION_TOKEN = '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'
DATABASE_ID= '334701e728cb8096a94cebc0985684a2'


headers = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Content-Type": "application/json",
    "Notion-Version": "2022-06-28",
}

def _safe_str(x, default=""):
    if x is None:
        return default
    s = str(x).strip()
    return s if s else default

def _safe_iso(dt_str: Optional[str]) -> Optional[str]:
    if not dt_str:
        return None
    try:
        _ = datetime.fromisoformat(dt_str.replace("Z", "+00:00"))
        return dt_str
    except Exception:
        return None

def create_page(properties: dict):
    url = "https://api.notion.com/v1/pages"
    payload = {"parent": {"database_id": DATABASE_ID}, "properties": properties}
    res = requests.post(url, headers=headers, json=payload, timeout=60)
    if not res.ok:
        print("Notion error:", res.status_code, res.text[:500])
        res.raise_for_status()
    return res

# 1) Load already-uploaded titles to avoid duplicates
csv_path = "contract_titles.csv"
uploaded_titles = set()
if os.path.exists(csv_path):
    try:
        prev = pd.read_csv(csv_path)
        if "Contract Name" in prev.columns:
            uploaded_titles = set(prev["Contract Name"].dropna().astype(str).str.strip())
    except Exception as e:
        print("Warning: could not read contract_titles.csv:", e)

# 2) Build payloads for titles not yet uploaded
now_iso = datetime.now(timezone.utc).isoformat()
to_upload = []
new_rows_for_csv = []
seen_this_run = set()

for _, r in df.iterrows():
    name = _safe_str(r.get("title"))
    if not name:
        continue
    if name in uploaded_titles or name in seen_this_run:
        continue

    # Compose Value: pretty amount + currency
    pretty_amount = _safe_str(r.get("value_amount"), "")   # already comma-formatted
    currency      = _safe_str(r.get("value_currency"), "")
    if pretty_amount and currency:
        value_display = f"{pretty_amount} {currency}"
    elif pretty_amount:
        value_display = pretty_amount
    elif currency:
        value_display = currency
    else:
        value_display = "Not Disclosed"

        # Enforce Notion character limits
    name = name[:1000]  # clamp title
    description_text = _safe_str(r.get("description_most_recent"), "Not Disclosed")[:2000]
    cpv_text = _safe_str(r.get("cpv_codes"), "")[:2000]
    contract_dates_text = _safe_str(r.get("contract_dates_text"), "")[:2000]
    client_text = _safe_str(r.get("employer_name"), "Not Disclosed")[:2000]
    language_text = _safe_str(r.get("language"), "")[:2000]
    locations_text = _safe_str(r.get("locations"), "")[:2000]
    value_display = value_display[:2000]
    closing_date_iso = _safe_iso(_safe_str(r.get("application_deadline_iso")))
    
    props = {
        "Name": {"title": [{"text": {"content": name}}]},
        "CPV Codes": {"rich_text": [{"text": {"content": cpv_text}}]},
        "Client": {"rich_text": [{"text": {"content": client_text}}]},
        "Contract Dates": {"rich_text": [{"text": {"content": contract_dates_text}}]},
        "Contract Link": {"url": _safe_str(r.get("find_a_tender_url")) or None},
        "Date Added": {"date": {"start": now_iso, "end": None}},
        "Closing Date": {"date": {"start": closing_date_iso, "end": None}} if closing_date_iso else {"date": None},
        "Description": {"rich_text": [{"text": {"content": description_text}}]},
        "Employer Website": {"url": _safe_str(r.get("employer_website")) or None},
        "Language": {"rich_text": [{"text": {"content": language_text}}]},
        "Location": {"rich_text": [{"text": {"content": locations_text}}]},
        "Reviewed By": {"select": {"name": "N/A"}},
        "Review Status": {"select": {"name": "Not Reviewed"}},
        "Value": {"rich_text": [{"text": {"content": value_display}}]},
        "Contract Status": {"select": {"name": "Open"}},
        "Source": {"select": {"name": "Find a Tender"}}
    }

    to_upload.append((name, props))
    seen_this_run.add(name)

# 3) Upload newest first (reverse to mimic your other flow)
to_upload.reverse()

for name, props in to_upload:
    try:
        create_page(props)
        new_rows_for_csv.append({"Contract Name": name})
    except Exception as e:
        print(f"Error creating Notion page for '{name}': {e}")

# 4) Record newly uploaded titles to CSV
if new_rows_for_csv:
    new_df = pd.DataFrame(new_rows_for_csv, columns=["Contract Name"])
    header_needed = not os.path.exists(csv_path) or \
        ("Contract Name" not in (pd.read_csv(csv_path).columns if os.path.exists(csv_path) else []))
    new_df.to_csv(csv_path, mode="a", header=header_needed, index=False)

print(f"Uploaded {len(new_rows_for_csv)} new Find a Tender contracts to Notion.")
# ---------- end Notion upload ----------


Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"00ae75a8-d5cc-41c4-90d3-c50a5028f1a3"}
Error creating Notion page for 'Evaluation of the Places of Worship Renewal Fund - UID289': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages


Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"54870dc3-ac95-4513-b194-5fb9830d3998"}
Error creating Notion page for 'The Practice Education Facilitator role in Return to Practice': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages
Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"fef55773-ec59-4028-b7be-98a1019b2430"}
Error creating Notion page for 'Evaluation of the Heritage at Risk Capital Fund - UID290': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages


Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"1ee93474-98ab-4fff-a37a-e070ef69f118"}
Error creating Notion page for 'Non-School Alternative Provision / Education other then in school or college - Social Emotional Mental Health (SEMH) and Vocational l Framework': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages
Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"03d87d4c-307c-4e93-bc91-3cb16a6f3da3"}
Error creating Notion page for 'Net Promoter Score': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages


Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"4e400bec-199f-46df-8b1b-44ae216fbcd6"}
Error creating Notion page for 'Adults Skills Qualifications - Lots 1-3': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages


Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"5fca8c2e-35f0-4945-9cca-392a08f6906f"}
Error creating Notion page for 'Assistive Support Needs Services for South Lanarkshire College': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages


Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"81633234-299a-4de3-8906-7605b892818f"}
Error creating Notion page for 'Strategic Culture Change Partner': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages


Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"6012efe0-6c7d-4acc-9a90-692f671c8f3e"}
Error creating Notion page for 'Higher Specialist Scientist Training Programme (HSST)': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages
Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"b080bbf5-93b8-4b14-adf5-8d4619dd3458"}
Error creating Notion page for 'Neurodiverse Environmental Audits': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages


Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"6ab544b5-9d1b-4b39-bf47-37d45cb4e896"}
Error creating Notion page for 'ET190 Consultancy - Capacity Strenthening': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages
Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"412ce3fe-c7ee-4c7b-80ed-8877cc8af096"}
Error creating Notion page for 'Effect of probiotics on Campylobacter colonisation within the broiler gut': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages


Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"6092a27b-f173-4e26-a623-393103b44e91"}
Error creating Notion page for 'RFQ 2026/25 - NMANDD PEACEPLUS Thriving Together Celebrating Cultures and Diversity 3.5 Sports Festival': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages


Notion error: 401 {"object":"error","status":401,"code":"unauthorized","message":"API token is invalid.","request_id":"74c7a834-0100-4a2a-bd4e-a07be0f28d68"}
Error creating Notion page for 'Exposure and Effects of Veterinary Medicines and Pesticides to Aquatic Wildlife in Protected Sites.': 401 Client Error: Unauthorized for url: https://api.notion.com/v1/pages
Uploaded 0 new Find a Tender contracts to Notion.
